# Averis X Monash Hackathon
**Team dareDEVils**

1. Classification Pipeline testing
2. Comparison & Analytics Strategy testing

Import all the required Libraries and Modules

In [ ]:
# for data loading
import pandas as pd
import os
import glob
import json

The data is to be retrieved either directly from data_v2 folder or using a uvicorn FastAPI server with endpoints:

**The endpoints**
 
| Method | Path | Returns |
|---|---|---|
| GET | `/health` | `{"status","emails","scoring_available"}` |
| GET | `/emails` | list of all 520 email records |
| GET | `/emails/{email_id}` | one record, e.g. `/emails/email_004` |
| GET | `/attachments/{path}` | the raw file bytes |
| GET | `/sample_submission` | the exact output shape, all 520 keys |
| POST | `/submit` | scoreboard JSON |
| GET | `/ground_truth` | 404 unless `REVEAL_GT=1` — judges only |
 
`{path}` is the attachment string **minus** the `attachments/` prefix already in
the URL, so `attachments/email_004_SI.txt` → `GET /attachments/email_004_SI.txt`.
Just concatenate: `base_url + "/" + att_string` gives the right URL either way.

The expected input is an email for the given dataset which haas 5 fields and is a json of the format:
```bash
{
  "email_id": "email_XXX",
  "from": "abc1234@pqrs.xxx",
  "subject": "XXXX YYYY ZZZZ",
  "body": "lorem ipsum ........",
  "attachments": ["attachments/email_XXX_SI.yyy", "attachments/email_XXX_BL.yyy"]
}
```

Load the input data directly from data_v2/inbox using pandas

In [ ]:
# List out all the filenames of the email json files in inbox directory
inbox_dir = "data_v2/inbox"
mail_files = os.path.join(inbox_dir, "*.json") # all mail as json files
mail_files_list = glob.glob(mail_files) # list of all json files

# for each mail file, read and add the mail information to dataframe
mail_data = []
for mail_file in mail_files_list:
    with open(mail_file, "r") as mfile:
        mail_data.append(pd.json_normalize(json.loads(mfile.read())))

# convert the extracted json fields data to dataframe
mail_df = pd.concat(mail_data)